# Where do recognition errors come from?

Two paired GPU ablations on FMA medium isolate different decisions. First, how many answers do we gain by passing frozen MuQ features through the trained fingerprint head? Second, does a spherical inverted-file index preserve the runner-up as well as the winning song? The first experiment uses the original full-catalogue test. The second revisits the earlier missing-song cohort. A separate fresh-cohort study evaluates the proposed acceptance rule.

Run all cells in order with Python 3 and pandas. The saved measurements travel with this notebook. The final cell notes explain how to regenerate them on CUDA. Companion notebooks cover the [exact and LSH comparison](https://github.com/nickiliak/02807_computational_tools_ds/pull/1), [original open-set protocol](https://github.com/nickiliak/02807_computational_tools_ds/pull/2), [temporal controls](https://github.com/nickiliak/02807_computational_tools_ds/pull/3), [spherical clustering](https://github.com/nickiliak/02807_computational_tools_ds/pull/4) and [fresh runner-up experiment](https://github.com/ExoHumann/02807_computational_tools_ds/tree/krzysztof/runner-up-abstention).

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'experiments/medium_ablations').is_dir())
data = root / 'experiments/medium_ablations'
projection = json.loads((data / 'projection-ablation.json').read_text())
routing = json.loads((data / 'cluster-margins.json').read_text())
print(f"{projection['device']} | {projection['reference_windows']:,} windows for the representation study")

NVIDIA GeForce RTX 4090 | 110,320 windows for the representation study


## Same audio, same exact search, different fingerprint

We match track IDs, window offsets, query labels and the silence filter before retrieval. Both representations use unit-normalised vectors and exhaustive GPU cosine search. The projected-head result reproduces the earlier exact totals. The intervals resample songs rather than treating repeat queries as independent. Changes in feature dimension and training occur together.

In [2]:
rows = []
for name, family in projection['families'].items():
    n = family['queries']
    assert family['trained_correct'] - family['raw_correct'] == family['gained'] - family['lost']
    lo, hi = family['paired_song_bootstrap95_pp']
    rows.append({'Family': {'queries': 'Standard', 'queries_tiktok': 'TikTok-like'}[name],
                 'Queries': n, 'Raw MuQ (%)': 100 * family['raw_correct'] / n,
                 'Trained (%)': 100 * family['trained_correct'] / n,
                 'Gain (pp)': family['delta_pp'], 'Paired 95% interval (pp)': f'[{lo:.2f}, {hi:.2f}]',
                 'Corrected': family['gained'], 'Lost': family['lost']})
display(pd.DataFrame(rows).set_index('Family').round(2))

,Queries,Raw MuQ (%),Trained (%),Gain (pp),Paired 95% interval (pp),Corrected,Lost
Family,,,,,,,
Standard,7458,92.79,98.08,5.30,"[4.75, 5.85]",417,22
TikTok-like,8992,58.02,93.54,35.52,"[34.52, 36.56]",3219,25


## Same fingerprints, different candidate lists

We removed all windows for the same predetermined missing songs and fitted 256 spherical clusters to the remainder. IVF searched the nearest 16 lists. The runner-up is the best-scoring window belonging to a different track. We measured both the identity of the winner and the identity and order of both best tracks. Each acceptance rule is calibrated on its own development unknowns to the same empirical 1% target. This is an exploratory analysis of an earlier test cohort.

In [3]:
n = routing['test_queries']
print(f"Exact and IVF agree on the winner for {routing['same_top']:,}/{n:,} queries; "
      f"on the ordered pair for {routing['same_ordered_top_two']:,}/{n:,}.")
rows = pd.DataFrame(routing['rows'])
own = rows[rows.retrieval == rows.calibration].copy()
own['Correct known (%)'] = 100 * own.correct_known / own.known_n
own['False unknown (%)'] = 100 * own.false_unknown / own.unknown_n
display(own[['retrieval','signal','correct_known','known_n','false_unknown','unknown_n',
             'Correct known (%)','False unknown (%)']].round(2).reset_index(drop=True))

Exact and IVF agree on the winner for 1,976/2,048 queries; on the ordered pair for 1,819/2,048.


,retrieval,signal,correct_known,known_n,false_unknown,unknown_n,Correct known (%),False unknown (%)
0,exact,best,515,1024,8,1024,50.29,0.78
1,ivf,best,515,1024,8,1024,50.29,0.78
2,exact,gap,691,1024,12,1024,67.48,1.17
3,ivf,gap,695,1024,12,1024,67.87,1.17


The trained projection changes identification much more than the observed differences between retrieval indexes. A margin based on clustered candidates can have a different value even when IVF gets the winner right. The observed false-acceptance rate for the gap exceeded 1% on this cohort. This measures recognition decisions; neither ablation measures search speed.

## Regenerate the GPU measurements

Run from the repository root. The [medium pipeline](https://github.com/ExoHumann/02807_computational_tools_ds/blob/5e9427ee5f83cd32fdd9b5a1c8f40396c3772989/README.md) produces the projected archives and cached clean/query MuQ features. The [original unknown-song study](https://github.com/ExoHumann/02807_computational_tools_ds/blob/d15e8e937472d5cb7b0f3597f58407aee9fc7d62/reports/unknown-songs.md) generates the matching unknown-song query archive. In a clone missing the two study commits, fetch the PR heads first.

~~~sh
git fetch https://github.com/nickiliak/02807_computational_tools_ds.git refs/pull/2/head refs/pull/4/head
# Medium PyTorch CUDA environment, with the original processed feature caches
python experiments/medium_ablations/check_projection.py .
# IVF CuPy CUDA environment, with the earlier unknown-song archive
python experiments/medium_ablations/check_cluster_margins.py .
~~~

The scripts check alignment and the previous winning predictions or counts before writing their JSON results. Then rerun this notebook to refresh the tables. GPU centroid sums can differ slightly between runs.